# 04 · Feature Extraction Timing
Profile `load_fixed` and `extract_features` on a stratified sample of 100 clips, then verify correctness with asserts.

In [6]:
# interpreter, path setup, imports
import sys
import time
from pathlib import Path

print('sys.executable:', sys.executable)

# Add repo root to sys.path so src.preprocessing.audio can be imported
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT added to sys.path:', REPO_ROOT)

import numpy as np
import pandas as pd
from src.preprocessing.audio import load_fixed, extract_features, SR, DURATION, N_MELS

print(f'Imported: SR={SR}, DURATION={DURATION}, N_MELS={N_MELS}')
print('Fixed clip length (samples):', SR * DURATION)

sys.executable: c:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT added to sys.path: c:\Users\LENOVO\Desktop\CityPulse
Imported: SR=22050, DURATION=4, N_MELS=64
Fixed clip length (samples): 88200


In [7]:
# load labels.csv; stratified sample of 100 (20 per app_class)
labels_path = REPO_ROOT / 'data' / 'processed' / 'labels.csv'
df = pd.read_csv(labels_path)
print('labels.csv shape:', df.shape)
print('app_class counts:')
print(df['app_class'].value_counts().to_string())

# Stratified sample: 20 rows per app_class → 100 rows total
# include_groups=False avoids pandas DeprecationWarning in newer versions
# and prevents the groupby key from being consumed into the index.
sample_parts = []
for cls, grp in df.groupby('app_class'):
    sample_parts.append(grp.sample(n=20, random_state=42))
sample = pd.concat(sample_parts).reset_index(drop=True)

print(f'\nSample shape: {sample.shape}')
print('Sample app_class counts:')
print(sample['app_class'].value_counts().to_string())

labels.csv shape: (6558, 5)
app_class counts:
app_class
construction         2080
traffic_horn         1509
hvac                 1000
music_loudspeaker    1000
siren                 969

Sample shape: (100, 5)
Sample app_class counts:
app_class
construction         20
hvac                 20
music_loudspeaker    20
siren                20
traffic_horn         20


In [8]:
# timing: load_fixed and extract_features over 100 clips
waveforms = []   # store for Cell 4 asserts
features  = []   # store for Cells 4 & 5

total_load = 0.0
total_feat = 0.0

for rel_path in sample['path']:
    abs_path = REPO_ROOT / rel_path

    # time load_fixed
    t0 = time.perf_counter()
    y = load_fixed(abs_path)
    t1 = time.perf_counter()
    total_load += t1 - t0
    waveforms.append(y)

    # time extract_features
    t2 = time.perf_counter()
    feat = extract_features(y)
    t3 = time.perf_counter()
    total_feat += t3 - t2
    features.append(feat)

n_clips = len(sample)
mean_load = total_load / n_clips
mean_feat = total_feat / n_clips
total_per_clip = (total_load + total_feat) / n_clips

print(f'=== Timing over {n_clips} clips ===')
print(f'Total  load_fixed       : {total_load:.3f} s')
print(f'Total  extract_features : {total_feat:.3f} s')
print(f'Mean   load_fixed/clip  : {mean_load*1000:.2f} ms')
print(f'Mean   extract_features : {mean_feat*1000:.2f} ms')
print(f'Mean   total/clip       : {total_per_clip*1000:.2f} ms')
projected_total_s = total_per_clip * len(df)
print(f'\nProjected total for {len(df)} clips: {projected_total_s/60:.2f} min')

=== Timing over 100 clips ===
Total  load_fixed       : 0.820 s
Total  extract_features : 1.009 s
Mean   load_fixed/clip  : 8.20 ms
Mean   extract_features : 10.09 ms
Mean   total/clip       : 18.29 ms

Projected total for 6558 clips: 2.00 min


In [9]:
# asserts on correctness
FIXED_LEN = SR * DURATION   # 88200
N_FEATURES = N_MELS * 2     # 128

# Assert every waveform has the exact fixed length
for i, y in enumerate(waveforms):
    assert y.shape == (FIXED_LEN,), \
        f'Waveform {i}: expected shape ({FIXED_LEN},), got {y.shape}'

# Assert every feature vector has shape (128,)
for i, feat in enumerate(features):
    assert feat.shape == (N_FEATURES,), \
        f'Feature {i}: expected shape ({N_FEATURES},), got {feat.shape}'

# Stack into matrix for joint checks
feat_matrix = np.stack(features)   # shape: (100, 128)

# Assert feature matrix shape
assert feat_matrix.shape == (n_clips, N_FEATURES), \
    f'Feature matrix: expected ({n_clips},{N_FEATURES}), got {feat_matrix.shape}'

# Assert dtype float32 for every vector (checked on the matrix)
assert feat_matrix.dtype == np.float32, \
    f'dtype: expected float32, got {feat_matrix.dtype}'

# Assert no NaN
n_nan = np.isnan(feat_matrix).sum()
assert n_nan == 0, f'NaN count: expected 0, got {n_nan}'

# Assert no Inf
n_inf = np.isinf(feat_matrix).sum()
assert n_inf == 0, f'Inf count: expected 0, got {n_inf}'

print('=== Cell 4 Asserts ===')
print(f'All {n_clips} waveforms have shape ({FIXED_LEN},)               ✓')
print(f'All {n_clips} feature vectors have shape ({N_FEATURES},)        ✓')
print(f'Feature matrix shape: {feat_matrix.shape}                       ✓')
print(f'dtype: {feat_matrix.dtype}                                      ✓')
print(f'NaN count: {n_nan}                                              ✓')
print(f'Inf count: {n_inf}                                              ✓')
print('ALL ASSERTS PASSED')

=== Cell 4 Asserts ===
All 100 waveforms have shape (88200,)               ✓
All 100 feature vectors have shape (128,)        ✓
Feature matrix shape: (100, 128)                       ✓
dtype: float32                                      ✓
NaN count: 0                                              ✓
Inf count: 0                                              ✓
ALL ASSERTS PASSED


In [10]:
# mean feature value per app_class (sanity check: classes look different)
sample_reset = sample.reset_index(drop=True)
means = (
    pd.DataFrame({'app_class': sample_reset['app_class'],
                  'feat_mean': feat_matrix.mean(axis=1)})
      .groupby('app_class')['feat_mean']
      .mean()
      .sort_values()
)
print('=== Mean feature value per app_class ===')
print('(overall mean of the 128 numbers, one row per class)')
print(means.round(4).to_string())

=== Mean feature value per app_class ===
(overall mean of the 128 numbers, one row per class)
app_class
siren               -18.7471
traffic_horn        -17.8180
music_loudspeaker   -15.4490
hvac                -14.9736
construction        -13.3330
